## Import Required Libraries

In [28]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.width', 1000)

## Load and Extract the Dataset

In [29]:
# Load the data
df = pd.read_csv('../data/911.csv', nrows=200000)  # adjust path
print("Shape:", df.shape)

# Keep only relevant columns
df_model = df[['title', 'timeStamp', 'twp', 'desc']].copy()
df_model.rename(columns={
    'title': 'incident_title',
    'timeStamp': 'timestamp',
    'twp': 'township',
    'desc': 'description'
}, inplace=True)

# Extract incident type (e.g., "EMS" from "EMS: BACK PAINS")
df_model['incident_type'] = df_model['incident_title'].apply(lambda x: x.split(':')[0].strip())

Shape: (200000, 9)


## Extract Basic Time Features

In [30]:
# Convert timestamp to datetime
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'], errors='coerce')

# Extract components
df_model['hour'] = df_model['timestamp'].dt.hour
df_model['day_of_week'] = df_model['timestamp'].dt.dayofweek   # 0=Monday
df_model['month'] = df_model['timestamp'].dt.month
df_model['is_weekend'] = (df_model['day_of_week'] >= 5).astype(int)

# Drop raw timestamp
df_model.drop('timestamp', axis=1, inplace=True)

## Create a Pseudo‑Severity Target (From Description)

In [31]:
def extract_people(desc):
    if pd.isna(desc):
        return 1
    match = re.search(r'(\d+)\s*(?:person|people|victim|patient|individual)', desc, re.I)
    if match:
        return int(match.group(1))
    if re.search(r'multiple|several', desc, re.I):
        return 3
    return 1

def has_keyword(desc, keywords):
    if pd.isna(desc):
        return 0
    return int(bool(re.search('|'.join(keywords), desc, re.I)))

# Apply keyword extraction
df_model['people_involved'] = df_model['description'].apply(extract_people)
df_model['has_injuries'] = df_model['description'].apply(lambda x: has_keyword(x, ['injure', 'wound', 'hurt', 'bleed', 'trauma', 'unconscious']))
df_model['has_fire'] = df_model['description'].apply(lambda x: has_keyword(x, ['fire', 'flame', 'burn', 'smoke']))
df_model['has_structural_damage'] = df_model['description'].apply(lambda x: has_keyword(x, ['structure', 'collapse', 'building', 'house', 'apartment']))
df_model['life_threat'] = df_model['description'].apply(lambda x: has_keyword(x, ['cardiac', 'arrest', 'shooting', 'stabbing', 'gunshot', 'unconscious', 'not breathing']))

# Combine into a severity score (1–4)
def compute_severity(row):
    score = 1
    if row['has_injuries']:
        score += 1
    if row['life_threat']:
        score += 2
    if row['has_fire'] and row['has_structural_damage']:
        score += 2
    elif row['has_fire']:
        score += 1
    if row['people_involved'] > 2:
        score += 1
    if row['people_involved'] > 5:
        score += 1
    return min(max(score, 1), 4)

df_model['severity'] = df_model.apply(compute_severity, axis=1)

# Check distribution
print(df_model['severity'].value_counts().sort_index())

severity
1    199551
2       447
3         2
Name: count, dtype: int64


## Define Features (X) and Target (y)

In [32]:
# Keep only the columns that will be used as predictors
X = df_model[['incident_type', 'township', 'hour', 'day_of_week', 'month', 'is_weekend']].copy()
y = df_model['severity'].copy()

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts().sort_index())

X shape: (200000, 6)
y distribution:
 severity
1    199551
2       447
3         2
Name: count, dtype: int64


## Feature Engineering

### Cyclic Encoding for Hour, Day_of_Week, Month

In [33]:
def add_cyclic_features(df):
    # Hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    # Day of week
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    # Month
    df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)
    # Drop original linear columns
    df.drop(['hour', 'day_of_week', 'month'], axis=1, inplace=True)
    return df

X = add_cyclic_features(X)

### Optimize Location (Township)

In [34]:
# Count occurrences
township_counts = X['township'].value_counts()
rare = township_counts[township_counts < 100].index
X['township'] = X['township'].apply(lambda x: 'Other' if x in rare else x)

## Split Data (80/20) with Stratification by Incident Type

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=X['incident_type']
)
print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 160000
Test size: 40000


## Define Preprocessing Pipelines

In [36]:
# Identify column types
numeric_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
categorical_cols = ['incident_type', 'township']

# Numerical pipeline: impute (just in case) and scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute missing with 'Unknown', then one‑hot encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

## Build One Model per Incident Type

In [37]:
# Inside your loop over incident types
for itype in incident_types:
    print(f"Training model for {itype}...")
    # Subset training data for this type
    mask = X_train['incident_type'] == itype
    X_type = X_train[mask].drop('incident_type', axis=1)
    y_type = y_train[mask]
    
    # Define column types for this type (incident_type is gone)
    numeric_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
    categorical_cols = ['township']   # only township remains categorical
    
    # Build preprocessor for this type
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preprocessor_type = ColumnTransformer([
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])
    
    # Create pipeline
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    pipe = Pipeline([
        ('preprocessor', preprocessor_type),
        ('classifier', clf)
    ])
    
    # Fit
    pipe.fit(X_type, y_type)
    models[itype] = pipe
    print(f"  Done. Samples: {len(X_type)}")

Training model for Fire...
  Done. Samples: 23821
Training model for Traffic...
  Done. Samples: 56382
Training model for EMS...
  Done. Samples: 79797


## Evaluate the Ensemble on Test Data

In [38]:
# Prepare test features (drop incident_type for prediction, but keep it for routing)
X_test_type = X_test['incident_type'].copy()
X_test_features = X_test.drop('incident_type', axis=1).copy()

# Apply cyclic encoding (already done, but we need to ensure same columns)
# Our X_test already has cyclic features, so we can just use it.

y_pred = []
for idx in X_test_features.index:
    itype = X_test_type.loc[idx]          # get incident type by index
    if itype not in models:
        # fallback to most common model
        itype = max(models.keys(), key=lambda k: len(models[k].named_steps['classifier'].tree_.n_node_samples))
    model = models[itype]
    row = X_test_features.loc[idx]        # get feature row
    row_df = pd.DataFrame([row])
    pred = model.predict(row_df)[0]
    y_pred.append(pred)

print(classification_report(y_test, y_pred))

KeyboardInterrupt: 

## Tie‑Breaking Logic for Multiple Incidents

In [ ]:
def prioritize_incidents(incident_list, models):
    """
    incident_list: list of dicts, each with keys:
        'incident_type', 'township', 'hour', 'day_of_week', 'month', 'is_weekend'
    Returns sorted list (highest priority first).
    """
    results = []
    for inc in incident_list:
        # Prepare input DataFrame
        df_inc = pd.DataFrame([inc])
        df_inc = add_cyclic_features(df_inc)  # same function as before
        # Remember: after add_cyclic_features, hour/day/month are dropped
        itype = inc['incident_type']
        if itype not in models:
            itype = max(models.keys(), key=lambda k: len(models[k].named_steps['classifier'].tree_.n_node_samples))
        model = models[itype]
        # Drop incident_type (not used by model)
        df_inc = df_inc.drop('incident_type', axis=1)
        pred = model.predict(df_inc)[0]
        probs = model.predict_proba(df_inc)[0]
        confidence = np.max(probs)
        results.append((pred, confidence, inc))
    
    # Sort: higher severity first, then higher confidence
    results.sort(key=lambda x: (-x[0], -x[1]))
    return [r[2] for r in results]

## Example Prediction for a New Incident

In [ ]:
new_incident = pd.DataFrame([{
    'incident_type': 'Traffic',
    'township': 'Abington',
    'hour': 17,
    'day_of_week': 2,
    'month': 6,
    'is_weekend': 0,
    'people_involved': 3,
    'has_injuries': 1,
    'has_fire': 0,
    'has_structural_damage': 0,
    'life_threat': 0
}])

pred = pipeline.predict(new_incident)[0]
proba = pipeline.predict_proba(new_incident)[0]
print(f"Predicted severity: {pred}")
print(f"Probabilities: {proba}")

Predicted severity: 1
Probabilities: [1. 0. 0. 0.]
